In [1]:
# Remove packages that are not needed for this workshop
%pip uninstall -y diffusers gradio gradio-client

# Install compatible libraries
%pip install -q -U \
    "transformers>=4.51,<5" \
    "huggingface-hub>=0.34,<1" \
    accelerate \
    sentence-transformers \
    faiss-cpu \
    ipywidgets



Found existing installation: gradio_client 2.6.0
Uninstalling gradio_client-2.6.0:
  Successfully uninstalled gradio_client-2.6.0
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import numpy as np
import pandas as pd
import torch
import faiss
import time

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("PyTorch:", torch.__version__)
print("GPU available:", torch.backends.mps.is_available())
print("FAISS:", faiss.__version__)

PyTorch: 2.14.0
GPU available: True
FAISS: 1.15.1


In [15]:
EMBEDDING_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(EMBEDDING_MODEL)

print("Embedding Model Dimension -> ",embedder.get_sentence_embedding_dimension())

Embedding Model Dimension ->  384


/var/folders/lv/p9vlph914d5554g_z5tcrsdm0000gn/T/ipykernel_91004/1484055190.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding Model Dimension -> ",embedder.get_sentence_embedding_dimension())


In [12]:
sentences = ["This is an example sentence", "Each sentence is converted", 'Hello World this is an embedding example!']

embeddings = embedder.encode(
    sentences,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Number of Sentences -> ", len(sentences))
print("Embeddings Shape -> ", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of Sentences ->  3
Embeddings Shape ->  torch.Size([3, 384])


In [13]:
embeddings

tensor([[ 0.0677,  0.0635,  0.0487,  ...,  0.0381,  0.0600, -0.0422],
        [ 0.0864,  0.1028,  0.0054,  ...,  0.0703,  0.0749, -0.0842],
        [ 0.0034, -0.0100, -0.0022,  ...,  0.0666,  0.0543,  0.0588]],
       device='mps:0')

In [4]:
# Sentence A: I love playing cricket.
# Sentence B: Cricket is my favourite sport.
# Sentence C: The weather is very hot today.
# Compute Euclidian Distance, Dot Product and Cosine Similarity

sentences = ["I love playing cricket", "Cricket is my favourite sport", "The weather is very hot today"]

embeddings = embedder.encode(
    sentences,
    convert_to_numpy=True,
    show_progress_bar=True
)


dot_product = np.dot(embeddings[2], embeddings[1])
euclidian_distance = np.linalg.norm(embeddings[2] - embeddings[1])
cosine_similarity = np.dot(embeddings[2], embeddings[1]) / (np.linalg.norm(embeddings[2]) * np.linalg.norm(embeddings[1]))

print("Dot Product -> ", dot_product)
print("Euclidian Distance -> ", euclidian_distance)
print("Cosine Similarity -> ", cosine_similarity)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Dot Product ->  0.12007312
Euclidian Distance ->  1.3265948
Cosine Similarity ->  0.12007312


In [5]:
documents = [
    "The university library closes at 10 PM.",
    "Students must maintain 75 percent attendance.",
    "The examination registration deadline is November 15.",
    "The hostel provides vegetarian and non-vegetarian meals.",
    "Scholarships are available for eligible students.",
    "The cricket team won the inter-college tournament.",
    "The computer science department offers AI courses.",
    "The university provides free Wi-Fi in the library.",
    "Students can borrow five books at a time.",
    "Hostel residents must return before 11 PM."
]

documnet_embeddings = embedder.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embeddings Shape -> ", documnet_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings Shape ->  (10, 384)


In [6]:
def cosine_similarity(x,y):

  numerator = np.dot(x,y)
  denominator = np.linalg.norm(x) * np.linalg.norm(y)

  if denominator == 0:
    return 0.0

  return numerator / denominator

sim_1_2 = cosine_similarity(documnet_embeddings[0], documnet_embeddings[1])

sim_1_3 = cosine_similarity(documnet_embeddings[0], documnet_embeddings[2])

print("Similarity between Document 1 and Document 2 -> ", sim_1_2)
print("Similarity between Document 1 and Document 3 -> ", sim_1_3)


Similarity between Document 1 and Document 2 ->  0.15265742
Similarity between Document 1 and Document 3 ->  0.32226226


In [7]:
def semantic_serch(query, documents, embeddings, top_k=3):
  query_embedding = embedder.encode(query)

  similarities = np.array([cosine_similarity(query_embedding, doc_embedding) for doc_embedding in embeddings])

  top_indices = np.argsort(similarities)[::-1][:top_k]

  result = []

  for i in top_indices:
    result.append({
        "document": documents[i],
        "similarity": similarities[i]
    })

  return result

In [8]:
queries = [
    "When does the university library close?",
    "What is the attendance requirement?",
    "What is the deadline for registration?",
]

for q in queries:
  print("Query -> ", q)

  results = semantic_serch(q, documents, documnet_embeddings)

  for r in results:
    print("Document -> ", r["document"])
    print("Similarity -> ", r["similarity"])
    print()

  print("******************************************")

Query ->  When does the university library close?
Document ->  The university library closes at 10 PM.
Similarity ->  0.8670937

Document ->  The university provides free Wi-Fi in the library.
Similarity ->  0.42155945

Document ->  Hostel residents must return before 11 PM.
Similarity ->  0.36943185

******************************************
Query ->  What is the attendance requirement?
Document ->  Students must maintain 75 percent attendance.
Similarity ->  0.6330825

Document ->  Hostel residents must return before 11 PM.
Similarity ->  0.38908535

Document ->  The examination registration deadline is November 15.
Similarity ->  0.37990284

******************************************
Query ->  What is the deadline for registration?
Document ->  The examination registration deadline is November 15.
Similarity ->  0.7124547

Document ->  Hostel residents must return before 11 PM.
Similarity ->  0.27636024

Document ->  The university library closes at 10 PM.
Similarity ->  0.22424908